In [1]:
# loading packages
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn import metrics
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn import config_context
from sklearn.metrics import make_scorer, mean_absolute_error
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
from sklearn.ensemble import RandomForestRegressor
consumption = pd.read_csv("../data/train_hh_gt.csv")
df = pd.read_csv("../data/train_hh_features.csv")
df = df.merge(consumption, how= "left", on=["hhid", "survey_id"], validate="1:1")
df["dep_ratio"] = (df.num_children5 + df.num_children10) / df.hsize # not the exact definition of dependency raiton, but a still a good indicaotr of dependdency ration in this case
df["no_adult"] = np.where((df.num_adult_female == 0) & (df.num_adult_male == 0), 1, 0)
df["adult_female_ratio"] = df.num_adult_female / df.hsize
df = df.drop(["survey_id", "strata", "com", "hhid", "num_children5", "num_children10", "num_children18", "num_elderly", "num_adult_male", "num_adult_female", "sector1d", "region1", "region2", "region3", "region4", "region5", "region6", "region7"], axis =1)
df.head()

,weight,utl_exp_ppp17,male,hsize,age,owner,water,toilet,sewer,elect,...,consumed4500,consumed4600,consumed4700,consumed4800,consumed4900,consumed5000,cons_ppp17,dep_ratio,no_adult,adult_female_ratio
0,75,594.80627,Female,1,75,Owner,Access,Access,Access,Access,...,No,Yes,Yes,Yes,Yes,No,25.258402,0.000000,1,0.000000
1,150,1676.27230,Female,2,61,Owner,Access,Access,Access,Access,...,No,No,Yes,Yes,No,No,16.996706,0.000000,0,0.500000
2,375,506.93719,Male,5,49,Owner,Access,Access,Access,Access,...,Yes,Yes,Yes,Yes,No,Yes,13.671848,0.000000,0,0.200000
3,375,824.61786,Male,5,58,Not owner,Access,Access,Access,No access,...,No,No,Yes,Yes,No,No,7.189475,0.000000,0,0.200000
4,525,351.47644,Male,7,57,Owner,Access,Access,Access,Access,...,Yes,No,Yes,Yes,Yes,No,12.308855,0.142857,0,0.285714


In [3]:
df.dtypes.value_counts()

str        63
float64     7
int64       4
Name: count, dtype: int64

In [4]:
categorical_vars = df.select_dtypes("str").columns
df[categorical_vars] = df[categorical_vars].apply(lambda var: var.astype("category").cat.codes)

df.head()

,weight,utl_exp_ppp17,male,hsize,age,owner,water,toilet,sewer,elect,...,consumed4500,consumed4600,consumed4700,consumed4800,consumed4900,consumed5000,cons_ppp17,dep_ratio,no_adult,adult_female_ratio
0,75,594.80627,0,1,75,1,0,0,0,0,...,0,1,1,1,1,0,25.258402,0.000000,1,0.000000
1,150,1676.27230,0,2,61,1,0,0,0,0,...,0,0,1,1,0,0,16.996706,0.000000,0,0.500000
2,375,506.93719,1,5,49,1,0,0,0,0,...,1,1,1,1,0,1,13.671848,0.000000,0,0.200000
3,375,824.61786,1,5,58,0,0,0,0,1,...,0,0,1,1,0,0,7.189475,0.000000,0,0.200000
4,525,351.47644,1,7,57,1,0,0,0,0,...,1,0,1,1,1,0,12.308855,0.142857,0,0.285714


In [5]:
x = df.drop(columns= ["cons_ppp17", "weight"]) # share_Secondary is not dropped
y = np.log(df.cons_ppp17)
sam_weight = df.weight
x_train, x_test, y_train, y_test, w_train, w_test = train_test_split(x, y, sam_weight, test_size= 0.2, stratify= df["urban"] , random_state= 1234 )

In [6]:
from scipy.stats import randint, uniform
cv = StratifiedKFold(n_splits= 5, shuffle= True, random_state= 1234)
folds = list(cv.split(x_train, x_train["urban"]))


# param_distributions = {
#     "max_depth": randint(5, 30),
#     "min_samples_leaf": randint(1, 50),
#     "max_features": uniform(0.1, 0.6),
# }

# search = RandomizedSearchCV(
#     RandomForestRegressor(
#         n_estimators=400,
#         max_features="sqrt",
#         random_state=0, 
#         n_jobs=-1),
#     param_distributions,
#     n_iter=15,
#     scoring="neg_root_mean_squared_error",
#     cv=3,
#     random_state=0,
# )


from scipy.stats import randint, uniform
from sklearn import config_context, metrics
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "max_depth": randint(5, 31),        # 5 through 50
    "max_features": uniform(0.1, 0.6),  # 0.1 up to 0.7
}

with config_context(enable_metadata_routing=True):
    rf = RandomForestRegressor(
        n_estimators=300,
        random_state=1234,
        n_jobs=1,
    ).set_fit_request(sample_weight=True)

    weighted_mae = metrics.make_scorer(
        metrics.mean_absolute_error,
        greater_is_better=False,
    ).set_score_request(sample_weight=True)

    rf_search = RandomizedSearchCV(
        estimator=rf,
        param_distributions=param_distributions,
        n_iter=30,
        scoring=weighted_mae,
        cv=folds,
        random_state=1234,
        n_jobs=-1,
        refit=True,
        error_score="raise",
    )
    rf_search.fit(x_train, y_train, sample_weight=w_train)

print("Best sampled parameters:", rf_search.best_params_)
print(f"Mean CV weighted MAE (log): {-rf_search.best_score_:.4f}")

Best sampled parameters: {'max_depth': 26, 'max_features': np.float64(0.3626366434042687)}
Mean CV weighted MAE (log): 0.2729


In [7]:
y_pred = rf_search.predict(x_train)
print(f"Random Forest Training R2: {metrics.r2_score(y_true= y_train, y_pred= y_pred, sample_weight=w_train):.2f}")
print(f"Random Forest Training MAE: {metrics.mean_absolute_error(y_true= y_train, y_pred= y_pred, sample_weight=w_train):.2f}")
print(f"Random Forest Training RMSE: {metrics.root_mean_squared_error(y_true= y_train, y_pred= y_pred, sample_weight=w_train):.2f}")

Random Forest Training R2: 0.96
Random Forest Training MAE: 0.07
Random Forest Training RMSE: 0.13


In [8]:
y_pred_test= rf_search.predict(x_test)
print(f"Random Forest Test R2 (log values): {metrics.r2_score(y_true= y_test, y_pred= y_pred_test, sample_weight=w_test):.2f}")
print(f"Random Forest Test MAE (log values): {metrics.mean_absolute_error(y_true= y_test, y_pred= y_pred_test, sample_weight=w_test):.2f}")
print(f"Random Forest Test RMSE (log values): {metrics.root_mean_squared_error(y_true= y_test, y_pred= y_pred_test, sample_weight=w_test):.2f}")

cons_actual = np.exp(y_test)
cons_pred = np.exp(y_pred_test)
print(f"Random Forest Test R2 (actual values): {metrics.r2_score(y_true= cons_actual, y_pred= cons_pred, sample_weight=w_test):.2f}")
print(f"Random Forest Test MAE (actual values): {metrics.mean_absolute_error(y_true= cons_actual, y_pred= cons_pred, sample_weight=w_test):.2f}")
print(f"Random Forest Test RMSE (actual values): {metrics.root_mean_squared_error(y_true= cons_actual, y_pred= cons_pred, sample_weight=w_test):.2f}")

Random Forest Test R2 (log values): 0.73
Random Forest Test MAE (log values): 0.27
Random Forest Test RMSE (log values): 0.35
Random Forest Test R2 (actual values): 0.60
Random Forest Test MAE (actual values): 3.15
Random Forest Test RMSE (actual values): 5.88


In [11]:
cv = StratifiedKFold(n_splits= 5, shuffle= True, random_state= 1234)
folds = list(cv.split(x_train, x_train["urban"]))

# param_grid = {
#     "n_estimators": [100, 200, 500],
#     "max_depth": [5, 10, 30, None],
#     "min_samples_leaf": [1, 5, 10],
#     "max_features": ["sqrt", 0.5, 1.0],
# }

#resulted: {'max_depth': 30, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 500}

param_grid = {
    "n_estimators": [500],
    "max_depth": [30],
    "min_samples_leaf": [1, 3],
    "max_features": ["log2", 0.35, 0.5],
}

with config_context(enable_metadata_routing=True):
    rf = RandomForestRegressor(
        random_state=1234,
        n_jobs=1,
    ).set_fit_request(sample_weight=True)

    weighted_mae = metrics.make_scorer(
        metrics.mean_absolute_error, greater_is_better=False
    ).set_score_request(sample_weight=True)

    rf_search = GridSearchCV(
        estimator=rf,
        param_grid= param_grid,
        scoring=weighted_mae,
        cv=folds,
        n_jobs = -1,
        refit=True,
        error_score="raise",
    )
    rf_search.fit(x_train, y_train, sample_weight=w_train)
    
print(rf_search.best_params_)

{'max_depth': 30, 'max_features': 0.35, 'min_samples_leaf': 1, 'n_estimators': 500}


In [12]:
y_pred_test= rf_search.predict(x_test)
print(f"Random Forest Test R2 (log values): {metrics.r2_score(y_true= y_test, y_pred= y_pred_test, sample_weight=w_test):.2f}")
print(f"Random Forest Test MAE (log values): {metrics.mean_absolute_error(y_true= y_test, y_pred= y_pred_test, sample_weight=w_test):.2f}")
print(f"Random Forest Test RMSE (log values): {metrics.root_mean_squared_error(y_true= y_test, y_pred= y_pred_test, sample_weight=w_test):.2f}")

cons_actual = np.exp(y_test)
cons_pred = np.exp(y_pred_test)
print(f"Random Forest Test R2 (actual values): {metrics.r2_score(y_true= cons_actual, y_pred= cons_pred, sample_weight=w_test):.2f}")
print(f"Random Forest Test MAE (actual values): {metrics.mean_absolute_error(y_true= cons_actual, y_pred= cons_pred, sample_weight=w_test):.2f}")
print(f"Random Forest Test RMSE (actual values): {metrics.root_mean_squared_error(y_true= cons_actual, y_pred= cons_pred, sample_weight=w_test):.2f}")

Random Forest Test R2 (log values): 0.73
Random Forest Test MAE (log values): 0.27
Random Forest Test RMSE (log values): 0.35
Random Forest Test R2 (actual values): 0.60
Random Forest Test MAE (actual values): 3.14
Random Forest Test RMSE (actual values): 5.89
